## LLM Implementation in LangGraph -- Basic

1. Call LangGraph, LangChain
2. OpenAI module from LangChain
3. Build a State for messages
4. Build a Chat Loop
5. Graph for the simple logic in LangGraph

In [1]:
from typing import TypedDict, List
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
CHAT_MODEL = "gpt-5-mini"

In [4]:
class AgentState(TypedDict):
    system_message: SystemMessage
    human_messages: List[HumanMessage]
    llm: ChatOpenAI
    ai_response: ChatOpenAI

In [5]:
def initiate_llm(state: AgentState) -> AgentState:
    """
    Call a LLM client to initiate via LangChain Openai
    """
    
    state["llm"] = ChatOpenAI(model=CHAT_MODEL)

    return state

In [6]:
def invoke_llm(state: AgentState) -> AgentState:
    """
    Process system and human messages and invoke LLM response
    """
    messages = [state["system_message"], state["human_messages"]]
    response = state["llm"].invoke(messages)

    state["ai_response"] = response

    return state

In [7]:
graph = StateGraph(AgentState)

graph.add_node("llm_initiator", initiate_llm)
graph.add_node("invoke_llm", invoke_llm) 

graph.add_edge(START, "llm_initiator")
graph.add_edge("llm_initiator", "invoke_llm")
graph.add_edge("invoke_llm", END)

agent = graph.compile()

In [10]:
while True:
    user_input = input("Enter: ")
    
    if user_input in ["bye", "exit"]:
        print("Talk soon!")
        break
        
    result = agent.invoke(
        {
            "system_message": SystemMessage("You are a helpful assistant."),
            "human_messages": HumanMessage(content=user_input)
        }
    )
    print(result["ai_response"].content)

Enter:  bey


Do you mean “bye” (goodbye) or “Bey” as in Beyoncé? 

If you meant goodbye: Bye — take care!  
If you meant Beyoncé or something else, tell me what you’d like to know.


Enter:  exit


Talk soon!
